# v18 SD 1.5 VAE: Pretrained vs Finetuned Reconstruction

Compares reconstruction quality of the **stock** Stable Diffusion 1.5 VAE against the **same VAE after finetuning on v18** (`configs/vae/train/presets/v18_sd15_vae_x8_256.yaml`).

Notes:
- v18 frames are single-channel `uint16` thermal `.npy`. We normalize with the repo's `raw_uint16_percentile` mapping to `[-1, 1]` (NOT a naive `/255`).
- SD 1.5's VAE is 3-channel; the repo `DiffusersAutoencoderAdapter` repeats 1->3 on encode and averages 3->1 on decode, so both models are evaluated exactly as during training.
- Latent downsample is x8 (256 -> 32).
- **Part 1** runs now with the pretrained VAE. **Part 2** auto-skips until a finetuned checkpoint exists at the path below.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch


def resolve_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not resolve repository root from the current working directory.")


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.core.normalization import (
    RAW_UINT16_PERCENTILE,
    norm_to_uint16,
    resize_and_normalize,
)
from src.core.data.datasets import NPYImageDataset
from src.models.vae import (
    build_vae_from_config,
    load_diffusers_vae_config,
    load_vae_weights,
)
from src.cli.train_vae import (
    _compute_psnr,
    _compute_ssim,
    _compute_raw_reconstruction_metrics,
    _stretch_for_vis,
)

print(f"REPO_ROOT = {REPO_ROOT}")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
DATA_ROOT = REPO_ROOT / "data" / "raw" / "v18"
SPLIT = "val"
NUM_SAMPLES = 4
IMAGE_SIZE = 256
NORMALIZATION_MODE = RAW_UINT16_PERCENTILE
MODEL_ID = "runwayml/stable-diffusion-v1-5"

# Matches model_dir in configs/vae/train/presets/v18_sd15_vae_x8_256.yaml
FINETUNED_CKPT = (
    REPO_ROOT
    / "artifacts" / "checkpoints" / "vae" / "vae_runs"
    / "v18_sd15_vae_x8_256" / "VAE" / "vae_best.pt"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device          = {device}")
print(f"data split      = {(DATA_ROOT / SPLIT)}")
print(f"finetuned ckpt  = {FINETUNED_CKPT}")
print(f"ckpt exists?    = {FINETUNED_CKPT.exists()}")

In [ ]:
# ── Load + normalize sample frames ───────────────────────────────────────────
dataset = NPYImageDataset(root_dir=str(DATA_ROOT / SPLIT), transform=None)
n = min(NUM_SAMPLES, len(dataset))
if n == 0:
    raise RuntimeError(f"No .npy frames under {(DATA_ROOT / SPLIT)!s}")

sample_names = [dataset.files[i] for i in range(n)]
x = torch.stack([
    resize_and_normalize(
        dataset[i],
        image_size=IMAGE_SIZE,
        normalization_mode=NORMALIZATION_MODE,
    )
    for i in range(n)
]).to(device)  # (n, 1, IMAGE_SIZE, IMAGE_SIZE) in [-1, 1]

print(f"Loaded {n} frames -> batch {tuple(x.shape)}  range=[{x.min():.3f}, {x.max():.3f}]")
for name in sample_names:
    print("  ", name)

In [ ]:
# ── Build pretrained SD 1.5 VAE (same adapter used in training) ───────────────
pretrained_cfg = load_diffusers_vae_config(MODEL_ID, subfolder="vae")
vae_pretrained = build_vae_from_config(pretrained_cfg, device=device)
vae_pretrained.eval()
print(type(vae_pretrained).__name__, "ready (pretrained)")

In [ ]:
# ── Reconstruction + metrics helper ──────────────────────────────────────────
@torch.no_grad()
def reconstruct(vae, x_in: torch.Tensor) -> dict:
    """Return recon (in [-1,1]), display tensors, and per-sample metrics."""
    recon, _mu, _sigma = vae(x_in)
    psnr = _compute_psnr(x_in, recon, data_range=2.0)
    ssim = _compute_ssim(x_in, recon, data_range=2.0)
    raw = _compute_raw_reconstruction_metrics(
        norm_to_uint16(x_in),  # raw uint16-scale target for the same input frame
        recon,
        normalization_mode=NORMALIZATION_MODE,
    )
    return {
        "recon": recon,
        "input_vis": _stretch_for_vis(x_in).cpu(),
        "recon_vis": _stretch_for_vis(recon).cpu(),
        "psnr": psnr.cpu(),
        "ssim": ssim.cpu(),
        "raw": raw,
    }


out_pre = reconstruct(vae_pretrained, x)
print(f"pretrained  mean PSNR={out_pre['psnr'].mean():.2f} dB  SSIM={out_pre['ssim'].mean():.4f}")

## Part 1 — Pretrained SD 1.5 VAE reconstruction

In [ ]:
def to_img(t):
    """(1,H,W) stretched tensor -> (H,W) numpy."""
    return t.squeeze(0).numpy()


fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
if n == 1:
    axes = np.expand_dims(axes, axis=0)

for row in range(n):
    orig = to_img(out_pre["input_vis"][row])
    rec = to_img(out_pre["recon_vis"][row])
    err = np.abs(rec - orig)

    axes[row, 0].imshow(orig, cmap="inferno")
    axes[row, 0].set_title(f"Original\n{sample_names[row]}", fontsize=8)
    axes[row, 1].imshow(rec, cmap="inferno")
    axes[row, 1].set_title(
        f"Pretrained recon\nPSNR={out_pre['psnr'][row]:.2f} dB  SSIM={out_pre['ssim'][row]:.3f}",
        fontsize=8,
    )
    axes[row, 2].imshow(err, cmap="magma")
    axes[row, 2].set_title("Absolute error", fontsize=8)
    for col in range(3):
        axes[row, col].axis("off")

fig.suptitle("v18 — stock SD 1.5 VAE reconstruction", y=1.0)
fig.tight_layout()
plt.show()

## Part 2 — Finetuned vs pretrained (auto-skips without a checkpoint)

In [ ]:
out_ft = None
if FINETUNED_CKPT.exists():
    vae_ft = build_vae_from_config(load_diffusers_vae_config(MODEL_ID, subfolder="vae"), device=device)
    load_vae_weights(vae_ft, str(FINETUNED_CKPT), strict=False, map_location=device)
    vae_ft.eval()
    out_ft = reconstruct(vae_ft, x)
    print(f"finetuned   mean PSNR={out_ft['psnr'].mean():.2f} dB  SSIM={out_ft['ssim'].mean():.4f}")
else:
    print(
        f"No finetuned checkpoint yet at:\n  {FINETUNED_CKPT}\n\n"
        "Run the finetune, then re-run this notebook:\n"
        "  bash scripts/train/vae_sd15_v18_256.sh\n"
        "or point FINETUNED_CKPT at a saved epoch, e.g. .../VAE/vae_epoch_10.pt"
    )

In [ ]:
if out_ft is not None:
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row in range(n):
        orig = to_img(out_pre["input_vis"][row])
        rec_pre = to_img(out_pre["recon_vis"][row])
        rec_ft = to_img(out_ft["recon_vis"][row])
        d_psnr = float(out_ft["psnr"][row] - out_pre["psnr"][row])
        d_ssim = float(out_ft["ssim"][row] - out_pre["ssim"][row])

        axes[row, 0].imshow(orig, cmap="inferno")
        axes[row, 0].set_title(f"Original\n{sample_names[row]}", fontsize=8)
        axes[row, 1].imshow(rec_pre, cmap="inferno")
        axes[row, 1].set_title(
            f"Pretrained\nPSNR={out_pre['psnr'][row]:.2f}  SSIM={out_pre['ssim'][row]:.3f}",
            fontsize=8,
        )
        axes[row, 2].imshow(rec_ft, cmap="inferno")
        axes[row, 2].set_title(
            f"Finetuned\nPSNR={out_ft['psnr'][row]:.2f} ({d_psnr:+.2f})  "
            f"SSIM={out_ft['ssim'][row]:.3f} ({d_ssim:+.3f})",
            fontsize=8,
        )
        for col in range(3):
            axes[row, col].axis("off")

    fig.suptitle("v18 — pretrained vs finetuned SD 1.5 VAE", y=1.0)
    fig.tight_layout()
    plt.show()
else:
    print("Skipped: no finetuned checkpoint.")

In [ ]:
# ── Aggregate summary over the sampled frames ────────────────────────────────
def _raw_means(out):
    raw = out["raw"]
    if raw is None:
        return float("nan"), float("nan")
    return float(raw["l1"]), float(raw["mse"])


pre_raw_l1, pre_raw_mse = _raw_means(out_pre)
print(f"Samples: {n}  (split={SPLIT})\n")
print(f"{'metric':<14}{'pretrained':>14}{'finetuned':>14}{'delta':>12}")

def _row(name, pre_val, ft_val, fmt="{:.4f}", higher_better=True):
    if ft_val is None:
        print(f"{name:<14}{fmt.format(pre_val):>14}{'-':>14}{'-':>12}")
        return
    delta = ft_val - pre_val
    print(f"{name:<14}{fmt.format(pre_val):>14}{fmt.format(ft_val):>14}{('{:+.4f}').format(delta):>12}")

if out_ft is not None:
    ft_raw_l1, ft_raw_mse = _raw_means(out_ft)
    _row("PSNR (dB)", float(out_pre["psnr"].mean()), float(out_ft["psnr"].mean()), "{:.2f}")
    _row("SSIM", float(out_pre["ssim"].mean()), float(out_ft["ssim"].mean()))
    _row("raw L1", pre_raw_l1, ft_raw_l1, "{:.2f}")
    _row("raw MSE", pre_raw_mse, ft_raw_mse, "{:.2f}")
else:
    _row("PSNR (dB)", float(out_pre["psnr"].mean()), None, "{:.2f}")
    _row("SSIM", float(out_pre["ssim"].mean()), None)
    _row("raw L1", pre_raw_l1, None, "{:.2f}")
    _row("raw MSE", pre_raw_mse, None, "{:.2f}")

### Reading the results
- **PSNR / SSIM** are computed in the `[-1, 1]` normalized domain (`data_range=2.0`), matching the training logs in TensorBoard.
- **raw L1 / MSE** are in the original uint16 sensor scale (via `norm_to_uint16`), so they are comparable to `eval/raw_recon_*` from training.
- Finetuning should raise PSNR/SSIM and lower raw L1/MSE vs the stock VAE; deltas are shown per-frame in the grid titles and aggregated in the table.
- To inspect a specific epoch instead of best, set `FINETUNED_CKPT` to `.../VAE/vae_epoch_N.pt` and re-run Part 2.
- Increase `NUM_SAMPLES` for a steadier estimate; clear outputs before committing (notebooks are kept output-free per `AGENTS.md`).